# Twitter Sentiment Analysis (SVM)

**Objective**: Build a sentiment classifier using Support Vector Machine (SVM).

## Table of Contents
1. [Data Loading & Preprocessing](#1-data-loading--preprocessing)
2. [Feature Engineering (TF-IDF)](#2-feature-engineering-tf-idf)
3. [Model Training (SVM)](#3-model-training-svm)
4. [Evaluation on Test Data](#4-evaluation-on-test-data)
5. [Comparison with Logistic Regression](#5-comparison-with-logistic-regression)
6. [Export Model](#6-export-model)

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import joblib
import os

# ML libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Settings
pd.set_option('display.max_colwidth', 100)
%matplotlib inline

---
## 1. Data Loading & Preprocessing

In [ ]:
# Load training data
train_df = pd.read_csv(
    'trainingandtestdata/training.1600000.processed.noemoticon.csv',
    encoding='latin-1',
    header=None,
    names=['sentiment', 'id', 'date', 'query', 'user', 'text']
)

# Load test data
test_df = pd.read_csv(
    'trainingandtestdata/testdata.manual.2009.06.14.csv',
    encoding='latin-1',
    header=None,
    names=['sentiment', 'id', 'date', 'query', 'user', 'text']
)

print(f"Training data: {len(train_df):,} samples")
print(f"Test data: {len(test_df)} samples")

In [ ]:
def preprocess_text(text):
    """
    Clean tweet text for sentiment analysis.
    """
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # URLs
    text = re.sub(r'@\w+', '', text)                      # @mentions
    text = re.sub(r'#', '', text)                         # # symbol
    text = re.sub(r'[^a-zA-Z\s]', '', text)               # special chars
    text = re.sub(r'\s+', ' ', text).strip()              # whitespace
    
    return text

In [ ]:
# Apply preprocessing
print("Preprocessing training data...")
train_df['clean_text'] = train_df['text'].apply(preprocess_text)
train_df['label'] = train_df['sentiment'].apply(lambda x: 1 if x == 4 else 0)
train_df = train_df[train_df['clean_text'].str.len() > 0]
print(f"Training size: {len(train_df):,}")

print("\nPreprocessing test data...")
test_df['clean_text'] = test_df['text'].apply(preprocess_text)
test_df = test_df[test_df['clean_text'].str.len() > 0]
print(f"Test size: {len(test_df)}")

---
## 2. Feature Engineering (TF-IDF)

In [ ]:
# Create TF-IDF vectorizer (same settings as logistic regression)
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

print("Fitting TF-IDF...")
X_train = tfidf.fit_transform(train_df['clean_text'])
y_train = train_df['label'].values

X_test = tfidf.transform(test_df['clean_text'])

print(f"Training matrix: {X_train.shape}")
print(f"Test matrix: {X_test.shape}")

---
## 3. Model Training (SVM)

**LinearSVC** (Linear Support Vector Classifier):
- Finds a hyperplane that maximizes the margin between classes
- Fast training on large datasets
- Does NOT output probabilities by default

To get probabilities for our neutral threshold, we use **CalibratedClassifierCV** which wraps the SVM and adds probability calibration.

In [ ]:
# Train LinearSVC with probability calibration
print("Training LinearSVC with probability calibration...")
print("(This may take a few minutes on 1.6M samples)\n")

# Base SVM model
base_svm = LinearSVC(
    C=1.0,
    max_iter=2000,
    random_state=42
)

# Wrap with CalibratedClassifierCV to get probabilities
# cv=3 means 3-fold cross-validation for calibration
svm_model = CalibratedClassifierCV(base_svm, cv=3)
svm_model.fit(X_train, y_train)

print("Training complete!")

In [ ]:
# Look at decision boundary (from the base estimators)
# Get feature importance from one of the calibrated estimators
base_estimator = svm_model.calibrated_classifiers_[0].estimator
feature_names = tfidf.get_feature_names_out()
coefficients = base_estimator.coef_[0]

word_importance = sorted(zip(feature_names, coefficients), key=lambda x: x[1])

print("Top 10 NEGATIVE words/phrases (SVM):")
for word, coef in word_importance[:10]:
    print(f"  {word:20} {coef:.4f}")

print("\nTop 10 POSITIVE words/phrases (SVM):")
for word, coef in word_importance[-10:]:
    print(f"  {word:20} {coef:.4f}")

---
## 4. Evaluation on Test Data

In [ ]:
# Define thresholds (same as logistic regression)
POSITIVE_THRESHOLD = 0.6
NEGATIVE_THRESHOLD = 0.4

def predict_sentiment(probability):
    if probability > POSITIVE_THRESHOLD:
        return 'positive'
    elif probability < NEGATIVE_THRESHOLD:
        return 'negative'
    else:
        return 'neutral'

In [ ]:
# Get predictions
y_proba_svm = svm_model.predict_proba(X_test)[:, 1]  # P(positive)
y_pred_svm = [predict_sentiment(p) for p in y_proba_svm]

# True labels
label_map = {0: 'negative', 2: 'neutral', 4: 'positive'}
y_true = test_df['sentiment'].map(label_map).values

# Store in dataframe
test_df['predicted_svm'] = y_pred_svm
test_df['probability_svm'] = y_proba_svm
test_df['true_label'] = y_true

In [ ]:
# Overall accuracy (3-class)
accuracy_3class = accuracy_score(y_true, y_pred_svm)
print(f"Overall Accuracy (3-class): {accuracy_3class:.2%}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred_svm, labels=['negative', 'neutral', 'positive']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred_svm, labels=['negative', 'neutral', 'positive'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - SVM')
plt.tight_layout()
plt.show()

In [ ]:
# Accuracy by class
print("Accuracy by Class (SVM):")
print("=" * 40)
for label in ['negative', 'neutral', 'positive']:
    subset = test_df[test_df['true_label'] == label]
    correct = (subset['true_label'] == subset['predicted_svm']).sum()
    total = len(subset)
    acc = correct / total if total > 0 else 0
    print(f"{label:10}: {acc:.2%} ({correct}/{total})")

In [ ]:
# Binary accuracy
binary_test = test_df[test_df['true_label'] != 'neutral']
binary_pred = binary_test['probability_svm'].apply(lambda p: 'positive' if p >= 0.5 else 'negative')
binary_acc = accuracy_score(binary_test['true_label'], binary_pred)

print(f"Binary Accuracy (pos/neg only): {binary_acc:.2%}")

---
## 5. Comparison with Logistic Regression

In [ ]:
# Train Logistic Regression for comparison
print("Training Logistic Regression for comparison...")
lr_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_model.fit(X_train, y_train)

# Get predictions
y_proba_lr = lr_model.predict_proba(X_test)[:, 1]
y_pred_lr = [predict_sentiment(p) for p in y_proba_lr]

test_df['predicted_lr'] = y_pred_lr
test_df['probability_lr'] = y_proba_lr

In [ ]:
# Compare accuracies
acc_svm_3class = accuracy_score(y_true, y_pred_svm)
acc_lr_3class = accuracy_score(y_true, y_pred_lr)

# Binary accuracy
binary_test = test_df[test_df['true_label'] != 'neutral']
acc_svm_binary = accuracy_score(
    binary_test['true_label'], 
    binary_test['probability_svm'].apply(lambda p: 'positive' if p >= 0.5 else 'negative')
)
acc_lr_binary = accuracy_score(
    binary_test['true_label'], 
    binary_test['probability_lr'].apply(lambda p: 'positive' if p >= 0.5 else 'negative')
)

print("Model Comparison:")
print("=" * 50)
print(f"{'Metric':<25} {'SVM':>10} {'Log. Reg.':>10}")
print("-" * 50)
print(f"{'3-class accuracy':<25} {acc_svm_3class:>10.2%} {acc_lr_3class:>10.2%}")
print(f"{'Binary accuracy':<25} {acc_svm_binary:>10.2%} {acc_lr_binary:>10.2%}")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 3-class accuracy
models = ['SVM', 'Logistic Regression']
acc_3class = [acc_svm_3class, acc_lr_3class]
colors = ['#2ecc71', '#3498db']

axes[0].bar(models, acc_3class, color=colors)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('3-Class Accuracy')
axes[0].set_ylim(0.5, 0.7)
for i, v in enumerate(acc_3class):
    axes[0].text(i, v + 0.01, f'{v:.2%}', ha='center')

# Binary accuracy
acc_binary = [acc_svm_binary, acc_lr_binary]
axes[1].bar(models, acc_binary, color=colors)
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Binary Accuracy (pos/neg only)')
axes[1].set_ylim(0.7, 0.9)
for i, v in enumerate(acc_binary):
    axes[1].text(i, v + 0.01, f'{v:.2%}', ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# Show cases where models disagree
disagree = test_df[test_df['predicted_svm'] != test_df['predicted_lr']]
print(f"Models disagree on {len(disagree)} samples ({len(disagree)/len(test_df)*100:.1f}%)")

print("\nSample disagreements:")
print("=" * 80)
for _, row in disagree.head(5).iterrows():
    print(f"Text: {row['text'][:60]}...")
    print(f"  True: {row['true_label']}")
    print(f"  SVM:  {row['predicted_svm']} (p={row['probability_svm']:.2f})")
    print(f"  LR:   {row['predicted_lr']} (p={row['probability_lr']:.2f})")
    print()

### Comparison Summary

| Model | Binary Acc | 3-Class Acc | Pros | Cons |
|-------|------------|-------------|------|------|
| **Logistic Regression** | ~80% | ~60% | Fast, native probabilities | - |
| **SVM (LinearSVC)** | ~80% | ~60% | Max-margin classifier | Needs calibration for probabilities |

Both models perform similarly on this dataset. Logistic Regression is simpler (no calibration needed), so it's a better choice for this project.

---
## 6. Export Model

Save the SVM model if you want to use it instead of Logistic Regression.

In [ ]:
# Save SVM model (optional - only if you want to use SVM)
os.makedirs('models', exist_ok=True)

# Uncomment to save SVM model:
# joblib.dump(tfidf, 'models/vectorizer_svm.joblib')
# joblib.dump(svm_model, 'models/model_svm.joblib')
# print("SVM model saved!")

print("Note: Logistic Regression model is recommended for this project.")
print("SVM provides similar accuracy but requires probability calibration.")

In [ ]:
# Test inference with SVM
def evaluate_svm(text):
    clean = preprocess_text(text)
    if not clean:
        return {'sentiment': 'neutral', 'confidence': 0.0, 'probability': 0.5}
    
    vec = tfidf.transform([clean])
    prob = svm_model.predict_proba(vec)[0][1]
    
    if prob > POSITIVE_THRESHOLD:
        sentiment = 'positive'
        confidence = prob
    elif prob < NEGATIVE_THRESHOLD:
        sentiment = 'negative'
        confidence = 1 - prob
    else:
        sentiment = 'neutral'
        confidence = 1 - abs(prob - 0.5) * 2
    
    return {
        'sentiment': sentiment,
        'confidence': round(confidence, 4),
        'probability': round(prob, 4)
    }

# Test
test_texts = [
    "I love this product!",
    "This is terrible.",
    "The meeting is at 3pm.",
]

print("SVM Predictions:")
print("=" * 60)
for text in test_texts:
    result = evaluate_svm(text)
    print(f"'{text}'")
    print(f"  -> {result}")
    print()

---
## Summary

### SVM Results
- **Binary accuracy**: ~80% (similar to Logistic Regression)
- **3-class accuracy**: ~60% (similar to Logistic Regression)

### Key Differences from Logistic Regression

| Aspect | Logistic Regression | SVM (LinearSVC) |
|--------|--------------------|-----------------|
| Training | Minimize log-loss | Maximize margin |
| Probabilities | Native | Requires calibration |
| Speed | Fast | Slightly slower (calibration) |
| Accuracy | ~80% | ~80% |

### Recommendation

**Use Logistic Regression** for this project because:
1. Similar accuracy to SVM
2. Native probability output (no calibration needed)
3. Simpler to explain and deploy